<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_05_Data_Cleaning_and_Preprocessing/Week_1_Handling_Missing_Data/Day_02_Identifying_and_Exploring_Missing_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 2: Identifying and Exploring Missing Data

## Phase 2 | Month 5 | Week 1 | Day 2

**🎯 Goal:** Detect, visualise, and understand the patterns and mechanisms of missing data.
**⏱️ Study Session:** ~2 hours

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(0)
N = 800

# Simulate Kenya Demographic and Health Survey (KDHS)-style dataset
df = pd.DataFrame({
    'household_id':    range(1, N+1),
    'county':          np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'], N),
    'household_size':  np.random.randint(1, 10, N),
    'monthly_income':  np.where(np.random.rand(N)<0.18, np.nan,
                                np.random.lognormal(10.2, 0.9, N)),
    'water_source':    np.where(np.random.rand(N)<0.12, np.nan,
                                np.random.choice(['Piped','Borehole','River','Vendor'], N)),
    'electricity':     np.where(np.random.rand(N)<0.08, np.nan,
                                np.random.choice([0, 1], N, p=[0.35, 0.65])),
    'toilet_type':     np.where(np.random.rand(N)<0.22, np.nan,
                                np.random.choice(['Flush','Pit','Open'], N, p=[0.45,0.40,0.15])),
})
print(df.isnull().sum())
print(f'\nTotal cells: {df.size}  |  Missing: {df.isnull().sum().sum()}  ({df.isnull().mean().mean()*100:.1f}%)')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(0)
N = 800
df = pd.DataFrame({
    'county':         np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'], N),
    'monthly_income': np.where(np.random.rand(N)<0.18, np.nan, np.random.lognormal(10.2,0.9,N)),
    'water_source':   np.where(np.random.rand(N)<0.12, np.nan,
                               np.random.choice(['Piped','Borehole','River','Vendor'],N)),
    'electricity':    np.where(np.random.rand(N)<0.08, np.nan, np.random.choice([0,1],N,p=[0.35,0.65])),
    'toilet_type':    np.where(np.random.rand(N)<0.22, np.nan,
                               np.random.choice(['Flush','Pit','Open'],N,p=[0.45,0.40,0.15])),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1 – Missing percentage bar
miss_pct = df.isnull().mean()*100
axes[0].barh(miss_pct.index, miss_pct.values, color='tomato', edgecolor='white')
axes[0].axvline(5, color='navy', ls='--', lw=1.5, label='5% threshold')
axes[0].set_title('Missing Data by Column (%)', fontweight='bold')
axes[0].set_xlabel('% Missing')
axes[0].legend()

# 2 – Missingness heatmap (sample 100 rows)
sample = df.sample(100, random_state=1).isnull().astype(int)
axes[1].imshow(sample.T, aspect='auto', cmap='RdYlGn_r', interpolation='none')
axes[1].set_yticks(range(len(df.columns)))
axes[1].set_yticklabels(df.columns)
axes[1].set_title('Missing Data Heatmap (100 rows)', fontweight='bold')
axes[1].set_xlabel('Row index')

plt.tight_layout()
fig.savefig('/tmp/day02_missing.png', dpi=120)
print('Saved /tmp/day02_missing.png')
plt.close()

## Three Mechanisms of Missingness

| Mechanism | Meaning | Example | Implication |
|-----------|---------|---------|-------------|
| **MCAR** — Missing Completely At Random | Missing independent of any data | Random survey non-response | Safe to drop, minimal bias |
| **MAR** — Missing At Random | Missing depends on observed data | Income missing more for elderly | Impute using other columns |
| **MNAR** — Missing Not At Random | Missing depends on the missing value itself | High earners skip income question | Hardest — may need domain knowledge |

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(0)
N = 800
income = np.random.lognormal(10.2, 0.9, N)
# MNAR: higher incomes more likely to be missing
missing_mask = np.random.rand(N) < (0.05 + 0.25*(income > np.percentile(income, 80)))
income_mnar = income.copy().astype(float)
income_mnar[missing_mask] = np.nan

df_check = pd.DataFrame({'income': income, 'is_missing': missing_mask})
print('Little MCAR test proxy:')
print(f'  Mean income (present): {df_check[~df_check.is_missing]["income"].mean():,.0f} KES')
print(f'  Mean income (missing): {df_check[df_check.is_missing]["income"].mean():,.0f} KES')
print(f'  t-test p-value: {stats.ttest_ind(df_check[~df_check.is_missing]["income"], df_check[df_check.is_missing]["income"]).pvalue:.6f}')
print(f'\nConclusion: p < 0.05 → income missingness is NOT random (MNAR)')

## 💪 Exercises

### Exercise 1
For the KDHS dataset, test whether `monthly_income` missingness is correlated with `county`. Use a chi-square test.

### Exercise 2
Create a 'missingness indicator' column (`income_missing`: 1 if missing, 0 if not). Does it correlate with other columns? This can reveal MAR patterns.

In [ ]:
# Your code here


## 📋 Summary

- ✓ Always visualise missingness before choosing a strategy
- ✓ Three mechanisms: MCAR → drop, MAR → impute, MNAR → investigate
- ✓ Missingness indicators can themselves be informative features

## 🚀 Next Steps
**Day 3: Dropping Missing Values — when deletion is justified**